# S9 Content-Based Model — LR vs EBM

**Prerequisite:** run `s9_foundation.ipynb` top-to-bottom in this kernel first, or let cell 1 auto-load it if you open this notebook alone.

**Features:** E4_Full from `feature_engineering_experiment.ipynb` (134 cols, `gender_B` dropped, includes `diff_age`), mean-imputed per S9 split.

This notebook:
1. Tunes LR (C grid) vs EBM on **S9 LOWO folds only**
2. Reduces E4_Full (134) → **95% cumulative LR importance** (LOWO fold-averaged |coef|, C=10)
3. Reports **AUC** on LOWO and wave-19 holdout (reduced features)
4. Compares **MM@5** / **NDCG@5** on `match` (unilateral + reciprocal HM)
5. Exports reduced feature list and unilateral scores

Requires: `pip install "interpret>=0.4"`


In [6]:
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss, roc_auc_score
from sklearn.preprocessing import StandardScaler

_FOUNDATION_REQUIRED = [
    "DATA", "S1_FEATS", "CB_FEATS", "THEME_MAP", "THEMES", "S9_SPLITS",
    "process_split", "ndcg_at5", "mutual_match_at5", "mirror_col",
    "add_reciprocal_scores", "ID_COLS", "RESULTS", "SEED", "CONFIG",
    "params_hash", "df_fingerprint", "DATA_SHA",
    "RAW_FEATURE_COLS", "EXCLUDED_FEATURE_COLS", "ENGINEERED_FEATURE_COLS", "FEATURE_GROUPS",
]


def _ensure_foundation() -> None:
    missing = [n for n in _FOUNDATION_REQUIRED if n not in globals()]
    if not missing:
        return

    here = Path.cwd()
    nb_path = None
    for candidate in [here, here / "notebooks", *here.parents[:4]]:
        p = candidate / "s9_foundation.ipynb"
        if p.exists():
            nb_path = p
            break
    if nb_path is None:
        raise RuntimeError(
            "s9_foundation.ipynb not found. Run it top-to-bottom first, "
            "or open this notebook from the repo/notebooks folder."
        )

    print(f"Loading foundation from {nb_path} ...")
    nb = json.loads(nb_path.read_text(encoding="utf-8"))
    for i, cell in enumerate(nb.get("cells", [])):
        if cell.get("cell_type") != "code":
            continue
        src = "".join(cell.get("source", []))
        if not src.strip():
            continue
        exec(compile(src, f"{nb_path.name}:cell{i}", "exec"), globals())

    still = [n for n in _FOUNDATION_REQUIRED if n not in globals()]
    if still:
        raise RuntimeError(f"Foundation load incomplete; still missing: {still}")
    print("Foundation loaded OK.")


_ensure_foundation()

try:
    import interpret
    from interpret.glassbox import ExplainableBoostingClassifier
    assert hasattr(ExplainableBoostingClassifier, "eval_terms")
    print(f"interpret {interpret.__version__} OK")
except ImportError as e:
    raise SystemExit(
        "interpret not installed. Run: pip install 'interpret>=0.4'"
    ) from e

print(f"S1_FEATS: {len(S1_FEATS)} features (E4_Full from feature_engineering_experiment)")


interpret 0.7.8 OK
S1_FEATS: 134 features (E4_Full from feature_engineering_experiment)


## 1 — Model definitions

In [7]:
def _to_theme(contrib_df: pd.DataFrame) -> pd.DataFrame:
    out = pd.DataFrame(0.0, index=contrib_df.index, columns=THEMES)
    for f in contrib_df.columns:
        out[THEME_MAP[f]] += contrib_df[f]
    return out


class Stage1LR:
    name = "lr"

    def __init__(self, C=1.0, seed=SEED):
        self.C = C
        self.scaler = StandardScaler()
        self.model = LogisticRegression(C=C, max_iter=2000, random_state=seed)

    def fit(self, train_df):
        X = self.scaler.fit_transform(train_df[S1_FEATS].to_numpy(float))
        self.model.fit(X, train_df["dec"])
        self.coef_ = pd.Series(self.model.coef_[0], index=S1_FEATS)
        self.intercept_ = float(self.model.intercept_[0])
        return self

    def predict_proba(self, df):
        X = self.scaler.transform(df[S1_FEATS].to_numpy(float))
        return self.model.predict_proba(X)[:, 1]

    def contributions(self, df):
        X = self.scaler.transform(df[S1_FEATS].to_numpy(float))
        c = X * self.model.coef_[0]
        logit = X @ self.model.coef_[0] + self.intercept_
        assert np.allclose(c.sum(1) + self.intercept_, logit, atol=1e-9)
        return pd.DataFrame(c, columns=S1_FEATS, index=df.index)

    def theme_contributions(self, df):
        return _to_theme(self.contributions(df))

    def describe(self):
        return {"model": "LogisticRegression", "C": self.C, "penalty": "l2",
                "features": list(S1_FEATS)}


class Stage1EBM:
    name = "ebm"

    def __init__(self, seed=SEED, **overrides):
        self.params = dict(
            interactions=0,
            outer_bags=14,
            learning_rate=0.01,
            max_bins=256,
            random_state=seed,
            n_jobs=-1,
        )
        self.params.update(overrides)
        self.model = ExplainableBoostingClassifier(**self.params)

    def fit(self, train_df):
        self.model.fit(train_df[S1_FEATS], train_df["dec"])
        self.intercept_ = float(np.ravel(self.model.intercept_)[0])
        return self

    def predict_proba(self, df):
        return self.model.predict_proba(df[S1_FEATS])[:, 1]

    def contributions(self, df):
        terms = self.model.eval_terms(df[S1_FEATS])
        c = pd.DataFrame(terms, columns=self.model.term_names_, index=df.index)
        assert set(c.columns) == set(S1_FEATS)
        c = c[list(S1_FEATS)]
        p_from_terms = 1.0 / (1.0 + np.exp(-(c.sum(1).to_numpy() + self.intercept_)))
        assert np.allclose(p_from_terms, self.predict_proba(df), atol=1e-6)
        return c

    def theme_contributions(self, df):
        return _to_theme(self.contributions(df))

    def describe(self):
        return {"model": "ExplainableBoostingClassifier", **self.params,
                "features": list(S1_FEATS)}


print("Models ready: lr, ebm")


Models ready: lr, ebm


## 2 — Hyperparameter tuning (S9 LOWO folds only)

In [ ]:
S9_LOWO = [sp for sp in S9_SPLITS if sp["kind"] == "lowo"]

CANDIDATES = {
    "lr_C0.1": lambda: Stage1LR(C=0.1),
    "lr_C1":   lambda: Stage1LR(C=1.0),
    "lr_C10":  lambda: Stage1LR(C=10.0),
    "ebm":     lambda: Stage1EBM(),
}

rows, theme_means = [], {}
for sp in S9_LOWO:
    tr, ev, _ = process_split(DATA, sp["train_waves"], sp["eval_waves"])
    for cand, ctor in CANDIDATES.items():
        t0 = time.time()
        m = ctor().fit(tr)
        p = m.predict_proba(ev)
        rows.append({
            "fold": sp["tag"], "cand": cand,
            "auc": roc_auc_score(ev["dec"], p),
            "logloss": log_loss(ev["dec"], p, labels=[0, 1]),
            "fit_s": round(time.time() - t0, 1),
        })
        theme_means[(cand, sp["tag"])] = m.theme_contributions(ev).mean()

h2h = pd.DataFrame(rows)
print(h2h.pivot(index="cand", columns="fold", values="auc").round(4).to_string())
summary = h2h.groupby("cand")[["auc", "logloss"]].agg(["mean", "std"]).round(4)
print("\n", summary.to_string())


def _min_cosine(cand):
    V = np.stack([theme_means[(cand, sp["tag"])].to_numpy() for sp in S9_LOWO])
    Vn = V / np.linalg.norm(V, axis=1, keepdims=True)
    cos = Vn @ Vn.T
    return cos[np.triu_indices_from(cos, k=1)].min()


stab = {c: round(float(_min_cosine(c)), 4) for c in CANDIDATES}
print("\nExplanation stability (min fold-pair cosine):", stab)

best_lr = h2h[h2h["cand"].str.startswith("lr")].groupby("cand")["auc"].mean().idxmax()
lr_auc = h2h[h2h["cand"] == best_lr]["auc"].mean()
ebm_auc = h2h[h2h["cand"] == "ebm"]["auc"].mean()
ebm_wins = (ebm_auc - lr_auc >= 0.005) and (stab["ebm"] >= 0.90)
winner = "ebm" if ebm_wins else best_lr

if winner == "ebm":
    CONFIG["s1_model"] = "ebm"
else:
    CONFIG["s1_model"] = "lr"
    CONFIG["s1_C"] = float(winner.split("C")[1])

print(f"\nbest LR: {best_lr} (AUC {lr_auc:.4f}) | EBM AUC {ebm_auc:.4f} | delta {ebm_auc - lr_auc:+.4f}")
print(f"WINNER -> {winner}")


## 3 — Feature reduction (95% cumulative LR importance)

Mirrors `feature_engineering_experiment.ipynb` Phase 2 Step 1, adapted for S9:
fold-averaged **|LR coefficient|** on **20 LOWO folds only** (wave 19 excluded), champion **lr_C10** (`C=10`), smallest prefix ≥ **95%** of normalized importance.

LR-only (no SHAP blend). Full 134-feature list kept as `FULL_S1_FEATS` for provenance.

In [ ]:
from s9_feature_engineering import select_reduced_features_lr, cumulative_coverage

IMPORTANCE_COVERAGE_TARGET = 0.95
IMPORTANCE_LR_C = 10.0

FULL_S1_FEATS = list(S1_FEATS)
FULL_CB_FEATS = list(CB_FEATS)

BASELINE_134 = {
    "lr": {
        "lowo_auc_mean": 0.6269,
        "lowo_auc_std": 0.0593,
        "holdout_auc": 0.6891,
        "train_ndcg_unilateral": 0.3244,
        "test_ndcg_unilateral": 0.4663,
        "train_mm_unilateral": 0.4136,
        "test_mm_unilateral": 0.6393,
    },
}

lr_coef_folds = []
for sp in S9_LOWO:
    tr, ev, _ = process_split(DATA, sp["train_waves"], sp["eval_waves"])
    m = Stage1LR(C=IMPORTANCE_LR_C).fit(tr)
    lr_coef_folds.append(m.coef_.abs())

REDUCED_FEATURES, importance_scores, selected_coverage = select_reduced_features_lr(
    lr_coef_folds, IMPORTANCE_COVERAGE_TARGET
)

S1_FEATS = list(REDUCED_FEATURES)
CB_FEATS = list(REDUCED_FEATURES)

eng_in_reduced = [f for f in REDUCED_FEATURES if f not in RAW_FEATURE_COLS]
print(f"Full E4_Full features: {len(FULL_S1_FEATS)}")
print(f"Importance coverage target: {IMPORTANCE_COVERAGE_TARGET:.0%}")
print(
    f"Selected {len(REDUCED_FEATURES)} features: {selected_coverage:.1%} cumulative LR importance "
    f"({len(REDUCED_FEATURES) / len(FULL_S1_FEATS):.1%} of E4_Full)"
)
print(f"Engineered retained ({len(eng_in_reduced)}): {eng_in_reduced}")

print("\n=== Top-10 by fold-averaged |coef| ===")
for i, (feat, score) in enumerate(importance_scores.head(10).items(), 1):
    cum = cumulative_coverage(importance_scores, i)
    print(f"  {i:2d}. {feat:<28} norm={score:.4f} cum={cum:.1%}")

print("\n=== Prefix coverage (selected size) ===")
for k in [5, 10, 20, 30, 40, 50, 60, 80, 100, len(REDUCED_FEATURES)]:
    if k > len(importance_scores):
        continue
    marker = " <-- selected" if k == len(REDUCED_FEATURES) else ""
    print(
        f"  top-{k:3d}: {cumulative_coverage(importance_scores, k):.1%} of importance"
        f"{marker}"
    )


## 4 — S9 full evaluation: AUC (target = `dec`)

In [ ]:
def make_champion():
    if CONFIG["s1_model"] == "ebm":
        return Stage1EBM()
    return Stage1LR(C=CONFIG["s1_C"])


EVAL_MODELS = {
    "lr": lambda: Stage1LR(C=float(best_lr.split("C")[1])),
    "ebm": lambda: Stage1EBM(),
}


def run_s9_model(model_ctor, model_name):
    parts = []
    for sp in S9_SPLITS:
        tr, ev, _ = process_split(DATA, sp["train_waves"], sp["eval_waves"])
        m = model_ctor().fit(tr)
        block = ev[ID_COLS].copy()
        block["score_cb"] = m.predict_proba(ev)
        block["split_tag"] = sp["tag"]
        block["split_kind"] = sp["kind"]
        block["model"] = model_name
        parts.append(block)
    out = pd.concat(parts, ignore_index=True).sort_values(["wave", "iid", "pid"])
    out["score_cb_rev"] = mirror_col(out, "score_cb")
    return out


scores = {}
auc_rows = []
for mname, ctor in EVAL_MODELS.items():
    s9 = run_s9_model(ctor, mname)
    scores[mname] = s9
    lowo = s9[s9["split_kind"] == "lowo"]
    hold = s9[s9["split_kind"] == "holdout"]
    lowo_aucs = [
        roc_auc_score(d["dec"], d["score_cb"])
        for _, d in lowo.groupby("split_tag")
        if d["dec"].nunique() == 2
    ]
    auc_rows.append({
        "model": mname,
        "lowo_auc_mean": float(np.mean(lowo_aucs)),
        "lowo_auc_std": float(np.std(lowo_aucs)),
        "holdout_auc": float(roc_auc_score(hold["dec"], hold["score_cb"])),
        "n_rows": len(s9),
    })

auc_table = pd.DataFrame(auc_rows)
print("=== AUC on dec (S9 protocol, reduced features) ===")
print(auc_table.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

print("\n=== vs 134-feature baseline (lr_C10, full E4_Full) ===")
for mname in ["lr"]:
    row = auc_table[auc_table["model"] == mname].iloc[0]
    base = BASELINE_134[mname]
    print(
        f"{mname}: LOWO {row['lowo_auc_mean']:.4f} (was {base['lowo_auc_mean']:.4f}, "
        f"delta {row['lowo_auc_mean'] - base['lowo_auc_mean']:+.4f}) | "
        f"holdout {row['holdout_auc']:.4f} (was {base['holdout_auc']:.4f}, "
        f"delta {row['holdout_auc'] - base['holdout_auc']:+.4f})"
    )


## 5 — Ranking metrics (relevance = `match`, k = 5)

In [ ]:
RANK_CONDITIONS = {
    "unilateral": "score_cb",
    "reciprocal_hm": "score_recip_hm",
}


def evaluate_ranking(df, split_name, model_name):
    df = add_reciprocal_scores(df)
    rows = []
    for cond, col in RANK_CONDITIONS.items():
        mm, n_users, n_excl = mutual_match_at5(df, col)
        ndcg, _ = ndcg_at5(df, col)
        rows.append({
            "split": split_name,
            "model": model_name,
            "condition": cond,
            "mm_at_5": mm,
            "ndcg_at_5": ndcg,
            "n_users": n_users,
            "n_excluded_no_match": n_excl,
            "n_rows": len(df),
        })
    return rows


rank_rows = []
for mname, s9 in scores.items():
    train = s9[s9["wave"] != 19]
    test = s9[s9["wave"] == 19]
    rank_rows.extend(evaluate_ranking(train, "train", mname))
    rank_rows.extend(evaluate_ranking(test, "test", mname))

rank_table = pd.DataFrame(rank_rows)
print("=== MM@5 and NDCG@5 (match relevance, reduced features) ===")
print(rank_table.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

lr_train = rank_table[(rank_table["model"] == "lr") & (rank_table["split"] == "train") & (rank_table["condition"] == "unilateral")].iloc[0]
lr_test = rank_table[(rank_table["model"] == "lr") & (rank_table["split"] == "test") & (rank_table["condition"] == "unilateral")].iloc[0]
base = BASELINE_134["lr"]
print("\n=== vs 134-feature baseline (lr, unilateral) ===")
print(
    f"train NDCG@5: {lr_train['ndcg_at_5']:.4f} (was {base['train_ndcg_unilateral']:.4f}) | "
    f"MM@5: {lr_train['mm_at_5']:.4f} (was {base['train_mm_unilateral']:.4f})"
)
print(
    f"test  NDCG@5: {lr_test['ndcg_at_5']:.4f} (was {base['test_ndcg_unilateral']:.4f}) | "
    f"MM@5: {lr_test['mm_at_5']:.4f} (was {base['test_mm_unilateral']:.4f})"
)


## 6 — Champion export: feature list + scores

In [ ]:
champion = make_champion()
contrib_cols = [f"contrib_{t}" for t in THEMES]

parts = []
for sp in S9_SPLITS:
    tr, ev, _ = process_split(DATA, sp["train_waves"], sp["eval_waves"])
    m = champion.fit(tr)
    block = ev[ID_COLS].copy()
    block["score_cb"] = m.predict_proba(ev)
    block[contrib_cols] = m.theme_contributions(ev).values
    block["split_tag"] = sp["tag"]
    block["split_kind"] = sp["kind"]
    parts.append(block)

s9_out = pd.concat(parts, ignore_index=True).sort_values(["wave", "iid", "pid"]).reset_index(drop=True)
s9_out["score_cb_rev"] = mirror_col(s9_out, "score_cb")

feat_parts = []
for sp in S9_SPLITS:
    _tr, ev, _ = process_split(DATA, sp["train_waves"], sp["eval_waves"])
    block = ev[ID_COLS + list(S1_FEATS)].copy()
    block["split_tag"] = sp["tag"]
    block["split_kind"] = sp["kind"]
    feat_parts.append(block)
features_out = pd.concat(feat_parts, ignore_index=True).sort_values(["wave", "iid", "pid"]).reset_index(drop=True)

assert len(s9_out) == 8368
assert len(features_out) == 8368

out_scores_pq = RESULTS / "stage1_s9_scores.parquet"
out_scores_csv = RESULTS / "stage1_s9_scores.csv"
out_feat_pq = RESULTS / "stage1_s9_features.parquet"
out_feat_csv = RESULTS / "stage1_s9_features.csv"

s9_out.to_parquet(out_scores_pq, index=False)
s9_out.to_csv(out_scores_csv, index=False)
features_out.to_parquet(out_feat_pq, index=False)
features_out.to_csv(out_feat_csv, index=False)

feature_list = {
    "n_features": len(S1_FEATS),
    "n_features_full": len(FULL_S1_FEATS),
    "source": "keep-w12 + feature_engineering_experiment E4_Full (LR-reduced)",
    "reduction": {
        "method": "LOWO fold-averaged |LR coef|, C=10, wave 19 excluded",
        "coverage_target": IMPORTANCE_COVERAGE_TARGET,
        "coverage_achieved": round(selected_coverage, 4),
        "lr_C": IMPORTANCE_LR_C,
    },
    "excluded_redundant": EXCLUDED_FEATURE_COLS,
    "engineered_columns": ENGINEERED_FEATURE_COLS,
    "engineered_in_reduced": eng_in_reduced,
    "features": [
        {"name": f, "theme": THEME_MAP[f]} for f in S1_FEATS
    ],
}
feat_json = RESULTS / "runs" / f"stage1_s9_feature_list_{time.strftime('%Y%m%d_%H%M%S')}.json"
feat_json.write_text(json.dumps(feature_list, indent=2), encoding="utf-8")

prov = {
    "ts": time.strftime("%Y-%m-%d %H:%M:%S"),
    "scheme": "S9",
    "feature_set": "E4_Full reduced by LR importance (95% cumulative)",
    "n_features": len(S1_FEATS),
    "n_features_full": len(FULL_S1_FEATS),
    "feature_reduction": feature_list["reduction"],
    "baseline_134_lr": BASELINE_134["lr"],
    "preprocessing": "train-fold mean impute; LR also standard-scaled on train",
    "winner": winner,
    "champion": champion.describe(),
    "auc": auc_table.to_dict("records"),
    "ranking": rank_table.to_dict("records"),
    "n_rows": len(s9_out),
    "data_sha": DATA_SHA,
    "config": CONFIG,
}
prov_path = RESULTS / "runs" / f"stage1_s9_summary_{time.strftime('%Y%m%d_%H%M%S')}.json"
prov_path.write_text(json.dumps(prov, indent=2, default=str), encoding="utf-8")

print(f"Champion: {CONFIG['s1_model']} | rows exported: {len(s9_out)}")
print(f"Saved scores: {out_scores_pq}")
print(f"Saved features: {out_feat_pq}")
